# 06b — KernelSHAP: prediction + explanation

**Interactive notebook** for explaining individual predictions using **KernelSHAP** on the full hybrid ensemble.

**Prereqs:** 
- Run **02_hybrid_ensemble.ipynb** (creates bundle)
- Run **06_kernel_shap.ipynb** (creates KernelExplainer background data)

**How it works:**
1. Enter property details (user input)
2. Build feature vector and predict price
3. Compute KernelSHAP values for the prediction
4. Visualize: waterfall plot, bar chart, feature contributions

**Note:** KernelSHAP is slow (~1-5 min per prediction) but provides **exact** SHAP values for the full hybrid model.

In [10]:
import importlib
import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

warnings_note = __import__("warnings")
warnings_note.filterwarnings("ignore")

_HERE = Path.cwd().resolve()


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "feature_data").is_dir() or (p / "hf_data").is_dir():
            return p
    return start.parent


REPO_ROOT = _repo_root(_HERE)

SNAPSHOT_DATE = "20260406"

_FEATURE_OUTPUT_CANDIDATES = [
    REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs",
    REPO_ROOT / "data" / "feature_data" / "02_feature_layer" / "training" / "outputs",
]


def _feature_outputs_dir() -> Path:
    for d in _FEATURE_OUTPUT_CANDIDATES:
        if d.is_dir() and any(d.glob("hdb_feature_table_*.csv")):
            return d
    tried = "\n  ".join(str(d) for d in _FEATURE_OUTPUT_CANDIDATES)
    raise FileNotFoundError(
        "No hdb_feature_table_*.csv found. Run notebooks/00_download_data_from_HF.ipynb "
        "or place CSVs under one of:\n  " + tried
    )


def _pick_feature_table(root: Path, snapshot_date: str | None) -> Path:
    if snapshot_date:
        p = root / f"hdb_feature_table_{snapshot_date}.csv"
        if p.exists():
            return p
        raise FileNotFoundError(f"Requested snapshot not found: {p}")
    tables = sorted(root.glob("hdb_feature_table_*.csv"))
    if not tables:
        raise FileNotFoundError(f"No hdb_feature_table_*.csv under {root}")
    return tables[-1]


HF_DATA_ROOT = _feature_outputs_dir()
YC_CSV = _pick_feature_table(HF_DATA_ROOT, SNAPSHOT_DATE)


def _hybrid_ml_dir(repo: Path) -> Path:
    candidates = [
        repo / "notebooks" / "03_ml_layer_hybrid",
        repo / "03_ml_layer_hybrid",
    ]
    for d in candidates:
        if (d / "yc_hybrid_inference.py").is_file():
            return d
    raise FileNotFoundError(
        "yc_hybrid_inference.py not found under notebooks/03_ml_layer_hybrid/."
    )


HYBRID_DIR = _hybrid_ml_dir(REPO_ROOT)
_XAI_LAYER = REPO_ROOT / "notebooks" / "04_xai_layer"
if not _XAI_LAYER.is_dir():
    _XAI_LAYER = REPO_ROOT / "04_xai_layer"
for _p in (HYBRID_DIR, _XAI_LAYER):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import yc_hybrid_inference as yhi
importlib.reload(yhi)
from yc_hybrid_inference import (
    build_yc_hybrid_vector,
    load_bundle,
    predict_price,
    USER_INPUT_KEYS,
)
from feature_labels import feature_label, labels_for_columns


def _hybrid_xai_out_dir(repo: Path) -> Path:
    candidates = [
        repo / "notebooks" / "03_ml_layer_hybrid" / "artifacts" / "hybrid_xai",
        repo / "data" / "artifacts" / "hybrid_xai",
    ]
    marker = "kernel_shap_values.joblib"
    for d in candidates:
        if (d / marker).is_file():
            return d
    tried = "\n  ".join(str(c / marker) for c in candidates)
    raise FileNotFoundError(
        "KernelSHAP artefacts not found. Run 06_kernel_shap.ipynb first. Tried:\n  " + tried
    )


OUT_DIR = _hybrid_xai_out_dir(REPO_ROOT)
print("OUT_DIR:", OUT_DIR)

# Load the SAME bundle that 06_kernel_shap.ipynb used to compute SHAP values.
# Prefer the notebook-local artifact; fall back to data/artifacts only if missing.
_LOCAL_BUNDLE = HYBRID_DIR / "artifacts" / "hybrid_cluster_bundle.joblib"
_DATA_BUNDLE = REPO_ROOT / "data" / "artifacts" / "hybrid_cluster_bundle.joblib"
BUNDLE_PATH = _LOCAL_BUNDLE if _LOCAL_BUNDLE.exists() else _DATA_BUNDLE
bundle = load_bundle(BUNDLE_PATH)
FEATURE_COLS = bundle["feature_columns"]
FEATURE_LABELS = labels_for_columns(FEATURE_COLS)

# Load KernelSHAP artifacts
kernel_shap_data = joblib.load(OUT_DIR / "kernel_shap_values.joblib")
background_data = kernel_shap_data["explainer_background"]
expected_value = kernel_shap_data["expected_value"]

# The bundle and the saved KernelSHAP background must share the same feature
# schema, otherwise predict_price / shap_values will shape-mismatch.
assert background_data.shape[1] == len(FEATURE_COLS), (
    f"Feature-count mismatch: bundle has {len(FEATURE_COLS)} features but "
    f"KernelSHAP background has {background_data.shape[1]}. Re-run "
    f"06_kernel_shap.ipynb against the same bundle loaded here ({BUNDLE_PATH})."
)

print(f"Bundle: {BUNDLE_PATH}")
print(f"Feature count: {len(FEATURE_COLS)}")
print(f"Background data shape: {background_data.shape}")
print(f"Expected value (base prediction): ${expected_value:,.0f}")

OUT_DIR: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai
Bundle: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_cluster_bundle.joblib
Feature count: 70
Background data shape: (50, 70)
Expected value (base prediction): $461,092


## 1 — User Input

Edit the property details below to get a prediction and explanation.

In [11]:
# Example user input (edit these values)
user_input = {
    "block": "1",
    "street_name": "LOR LEW LIAN",
    "town": "BEDOK",
    "flat_type": "4 ROOM",
    "floor_area_sqm": 90.0,
    "storey_range": "07 TO 09",
    "lease_commence_date": 1980,
    "sale_month": "2025-03",
}

# Build feature vector — pass FEATURE_COLS from the bundle so the vector is
# guaranteed to match the bundle's schema (avoids 70- vs 77-column drift when
# both notebook-local and data/artifacts bundles are present on disk).
built = build_yc_hybrid_vector(
    user_input["block"],
    user_input["street_name"],
    user_input["town"],
    user_input["flat_type"],
    user_input["floor_area_sqm"],
    user_input["storey_range"],
    user_input["lease_commence_date"],
    user_input["sale_month"],
    yc_csv=str(YC_CSV),
    feature_columns=FEATURE_COLS,
)
X = built["vector"]

# Predict price
price = float(predict_price(X, bundle)[0])

print(f"Predicted resale price (SGD): {price:,.2f}")
print(f"Lookup matched: {built['lookup_matched']} | {built['matched_address_key']}")
if built["imputation_note"]:
    print(f"Note: {built['imputation_note']}")

Predicted resale price (SGD): 545,377.04
Lookup matched: False | None
Note: No address_key match; using town-level medians for continuous POI/market fields.


## 2 — KernelSHAP Explanation

Computing SHAP values using KernelExplainer on the full hybrid model. This may take 1-5 minutes.

In [12]:
def hybrid_predict_fn(X_input):
    """Wrapper for predict_price that handles array conversion."""
    return predict_price(np.asarray(X_input, dtype=float), bundle)


# Create KernelExplainer with pre-computed background
print("Creating KernelExplainer...")
explainer = shap.KernelExplainer(hybrid_predict_fn, background_data)

# Compute SHAP values for this prediction
N_SAMPLES = 500  # Increase for more accuracy, decrease for speed
print(f"Computing SHAP values (nsamples={N_SAMPLES})...")
print("This may take 1-5 minutes...")

t0 = time.time()
shap_values = explainer.shap_values(X, nsamples=N_SAMPLES, silent=True)
elapsed = time.time() - t0

print(f"\nCompleted in {elapsed:.1f}s")
print(f"Base value: ${expected_value:,.0f}")
print(f"Sum of SHAP values: ${shap_values[0].sum():,.0f}")
print(f"Base + SHAP sum = ${expected_value + shap_values[0].sum():,.0f}")
print(f"Actual prediction = ${price:,.0f}")

Creating KernelExplainer...
Computing SHAP values (nsamples=500)...
This may take 1-5 minutes...


KeyboardInterrupt: 

## 3 — Top Feature Contributions

Features ranked by absolute SHAP value (impact on prediction).

In [ ]:
# Create DataFrame of contributions
sv = shap_values[0]
contributions = pd.DataFrame({
    "feature": FEATURE_COLS,
    "label": FEATURE_LABELS,
    "value": X[0],
    "shap_value": sv,
    "abs_shap": np.abs(sv),
}).sort_values("abs_shap", ascending=False)

print("Top 15 feature contributions:")
print("-" * 70)
for i, row in contributions.head(15).iterrows():
    sign = "+" if row["shap_value"] >= 0 else ""
    print(f"  {row['label']:40s} {sign}${row['shap_value']:>10,.0f}  (value={row['value']:.2f})")

## 4 — Visualizations

### 4.1 Waterfall Plot

Shows how each feature pushes the prediction up or down from the base value.

In [ ]:
# Create SHAP Explanation object for visualization
explanation = shap.Explanation(
    values=sv,
    base_values=expected_value,
    data=X[0],
    feature_names=FEATURE_LABELS,
)

# Waterfall plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.waterfall_plot(explanation, show=False, max_display=12)
plt.title(f"KernelSHAP Explanation: ${price:,.0f}")
plt.tight_layout()
plt.show()

### 4.2 Bar Plot (Magnitude)

Shows absolute impact of each feature (|SHAP|).

In [ ]:
# Bar plot of top features by magnitude
top_n = 15
top_contrib = contributions.head(top_n)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2d7d6b' if v >= 0 else '#e07a5f' for v in top_contrib['shap_value']]
y_pos = np.arange(len(top_contrib))
ax.barh(y_pos, top_contrib['shap_value'], color=colors, alpha=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_contrib['label'])
ax.invert_yaxis()
ax.set_xlabel('SHAP Value (SGD)')
ax.set_title(f'KernelSHAP Feature Contributions (Prediction: ${price:,.0f})')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5 — Explanation Summary

Natural language summary of why this property received this price prediction.

In [ ]:
# Generate natural language summary
top_positive = contributions[contributions['shap_value'] > 0].head(3)
top_negative = contributions[contributions['shap_value'] < 0].head(3)

print(f"Prediction: ${price:,.0f}")
print(f"Base value (average): ${expected_value:,.0f}")
print()

if len(top_positive) > 0:
    print("Factors INCREASING the price:")
    for _, row in top_positive.iterrows():
        print(f"  + {row['label']}: +${row['shap_value']:,.0f}")

print()

if len(top_negative) > 0:
    print("Factors DECREASING the price:")
    for _, row in top_negative.iterrows():
        print(f"  - {row['label']}: ${row['shap_value']:,.0f}")

print()
print(f"Net effect: ${sv.sum():+,.0f} from base")

## 6 — Notes

**KernelSHAP vs TreeSHAP:**
- KernelSHAP explains the **full hybrid** model (Ridge + XGB + LGB + RF + meta-learner)
- TreeSHAP (in `05_hybrid_xai_explain.ipynb`) only explains the XGBoost component
- KernelSHAP is slower but more accurate for the complete prediction

**To explain another property:**
1. Edit the `user_input` dictionary in Section 1
2. Re-run all cells from Section 1 onwards

**Speed tips:**
- Decrease `N_SAMPLES` (e.g., 100) for faster but less accurate results
- For production use, consider `07b_composite_treeshap_explain.ipynb` (faster)